# FAISS, Weaviate, Qdrant

## FAISS
Ключевые особенности:
- Скорость: Написана на C++ и отлично оптимизирована.
- Простота: Минимум настроек, быстрый старт, работает напрямую с массивами NumPy.
- Локальность: Работает внутри вашего Python-процесса, не требуя поднятия тяжелых баз данных (как PostgreSQL или Pinecone).
- Гибкость: Поддерживает десятки алгоритмов индексации (включая графовые HNSW, IVF, PQ и др.).

> Ограничение: «Чистый» FAISS не умеет хранить метаданные (текст, ID пользователя, ссылки). Он хранит только векторы и возвращает их порядковые номера

```python
# Версия для CPU
uv add faiss-cpu

# Версия для GPU с CUDA (на Windows могут быть проблемы, проще использовать CPU)
uv add faiss-gpu
```

In [ ]:
import faiss
import numpy as np

# 1. Подготовка данных (допустим, у нас 1000 документов, размерность вектора 128)
d = 128
nb = 1000
xb = np.random.random((nb, d)).astype("float32")  # База данных

# 2. Создаем индекс (простой перебор Flat по расстоянию L2)
index = faiss.IndexFlatL2(d)

# 3. Добавляем векторы в индекс
index.add(xb)
print(f"В индексе объектов: {index.ntotal}")

# 4. Поиск (ищем 3 ближайших вектора для одного нового запроса)
xq = np.random.random((1, d)).astype("float32")
k = 3
distances, indices = index.search(xq, k)

print("Индексы найденных векторов:", indices)
print("Расстояния до них:", distances)

В индексе объектов: 1000
Индексы найденных векторов: [[290 176 822]]
Расстояния до них: [[14.771656 14.905274 15.535244]]


### FAISS. Интеграция в LangChain

In [ ]:
from langchain_community.vectorstores import FAISS

# Создание хранилища: FAISS сам создаст индекс и свяжет векторы с текстом
vector_store = FAISS.from_documents(splitted_docs, embed_model)

# Поиск похожих документов
query = "Когда основан МГУ?"
docs_found = vector_store.similarity_search(query, k=2)

for doc in docs_found:
    print(f"Метаданные: {doc.metadata}")
    print(f"Контент: {doc.page_content[:100]}...")

Более гибкий способ (через инициализацию класса). Вы сначала создаете «пустой» объект индекса FAISS, настраиваете его внутренние параметры, а затем оборачиваете его в LangChain.

In [ ]:
import faiss
import numpy as np
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings  # Или любая другая модель

# --- 1. Подготовка параметров ---
embed_model = OpenAIEmbeddings()

# Узнаём размерность вектора
# Она зависит от модели (например, для text-embedding-3-small это 1536)
embedding_dim = len(embed_model.embed_query("hello world"))

# Параметры HNSW (графового индекса)
m_hnsw = 32  # Количество связей у каждого узла (обычно от 16 до 64)
ef_construction = 200  # Глубина поиска при создании графа (влияет на время индексации)
ef_search = 128  # Глубина поиска при самом запросе (влияет на скорость поиска)

# --- 2. Создание и настройка индекса FAISS ---
# Создаем индекс HNSWFlat (графовый индекс без сжатия векторов)
index = faiss.IndexHNSWFlat(embedding_dim, m_hnsw)

# Тонкая настройка точности
index.hnsw.efConstruction = ef_construction
index.hnsw.efSearch = ef_search

# --- 3. Инициализация обертки LangChain ---
vector_store = FAISS(
    embedding_function=embed_model,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

# Теперь можно добавлять документы
vector_store.add_documents(splitted_docs)

## Weaviate — облачная векторная БД

uv add weaviate-client

In [ ]:
import weaviate
import weaviate.classes.config as Configure
from sentence_transformers import SentenceTransformer

# 1. Инициализируем модель
model = SentenceTransformer("BAAI/bge-m3")

# 2. Подключение к локальному Weaviate (обычно запущен в Docker)
client = weaviate.connect_to_local()

try:
    # 3. Создаем коллекцию с описанием свойств
    client.collections.create(
        name="Laptop",
        properties=[
            Configure.Property(name="name", data_type=Configure.DataType.TEXT),
            Configure.Property(name="description", data_type=Configure.DataType.TEXT),
            Configure.Property(name="price", data_type=Configure.DataType.NUMBER),
        ],
    )

    # 4. Подготовка данных
    laptops = [
        {
            "name": "Dell XPS 13",
            "description": "Легкий ультрабук с OLED экраном для дизайнеров",
            "price": 150000,
        },
        {
            "name": "Игровой монстр",
            "description": "Мощная станция с RTX 4090 и водяным охлаждением",
            "price": 300000,
        },
    ]

    # 5. Добавление данных с ручной вставкой векторов
    collection = client.collections.get("Laptop")

    with collection.batch.dynamic() as batch:
        for item in laptops:
            # Генерируем вектор из описания
            vector = model.encode(item["description"]).tolist()
            batch.add_object(properties=item, vector=vector)

    # 6. Семантический поиск "компьютер для тяжелых игр"
    query_text = "компьютер для тяжелых игр"
    query_vector = model.encode(query_text).tolist()

    response = collection.query.near_vector(
        near_vector=query_vector, limit=1, return_properties=["name", "price"]
    )

    for obj in response.objects:
        print(f"Результат: {obj.properties['name']} за {obj.properties['price']} руб.")

finally:
    client.close()

Выбор модели. Мы использовали модель BAAI/bge-m3. Это Достаточно хороший выбор для мультиязычных задач (включая русский). Она поддерживает длинные тексты и дает очень высокую точность поиска. Можно использовать и другие, например OpenAI, тогда при создании  указываем, что Weaviate сам будет делать векторы через OpenAI. Например так:

In [ ]:
import weaviate
import weaviate.classes.config as Configure

# 1. Подключение к локальному серверу (обычно в Docker)
client = weaviate.connect_to_local()

try:
    # 2. Создание коллекции (схемы данных)
    client.collections.create(
        name="Articles",
        # Указываем, что Weaviate сам будет делать векторы через OpenAI
        vectorizer_config=Configure.Vectorizer.text2vec_openai(),
        properties=[
            weaviate.classes.config.Property(
                name="title", data_type=weaviate.classes.config.DataType.TEXT
            ),
            weaviate.classes.config.Property(
                name="content", data_type=weaviate.classes.config.DataType.TEXT
            ),
        ],
    )

    # 3. Добавление данных
    articles = client.collections.get("Articles")
    articles.data.insert(
        {
            "title": "МГУ",
            "content": "Московский государственный университет основан в 1755 году.",
        }
    )

    # 4. Семантический поиск (близость по смыслу, а не по словам)
    response = articles.query.near_text(
        concepts=["Когда открыли главный ВУЗ?"], limit=1
    )

    for obj in response.objects:
        print(obj.properties["title"], obj.properties["content"])

finally:
    client.close()

### Weaviate. Интеграция в LangChain

In [ ]:
import weaviate
from langchain_weaviate.vectorstores import WeaviateVectorStore

# Создаем клиент подключения
client = weaviate.connect_to_local()

# Создание хранилища из документов
vector_store = WeaviateVectorStore.from_documents(
    splitted_docs,
    embed_model,
    client=client,
    index_name="MyDocumentIndex",
    text_key="text",  # ключ, где хранится текст документа
)

# Поиск с фильтрацией (чего не умеет чистый FAISS)
where_filter = {
    "path": ["source"],  # имя поля в Weaviate
    "operator": "Equal",
    "valueString": "wikipedia",
}

docs = vector_store.similarity_search("История университета", k=2, filter=where_filter)

FAISS — мощная библиотека «в оперативной памяти». Она идеальна, когда вам нужна максимальная скорость поиска и минимальные накладные расходы. Однако она требует ручного управления текстами (через Docstore в LangChain) и сложна в настройке, если вам нужны фильтры по метаданным.

Weaviate — полноценная векторная база данных. Она позволяет хранить объекты (ноутбуки, статьи, пользователей) как в классической БД, привязывая к ним векторы. Это выбор для систем, где важны надежность, хранение на диске и сложные запросы (например, «найти по смыслу, но в категории до 100к рублей»).



## Qdrant

```shell
docker run -p 6333:6333 qdrant/qdrant
uv run qdrant-client
```

Коллекции в Qdrant — основная единица хранения данных. Каждая коллекция содержит векторы одной размерности и может иметь свои настройки индексации.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

client = QdrantClient(url="http://localhost:6333")

# 1. Создаем коллекцию (нужно сделать один раз)
# recreate_collection удалён в новых версиях, используем delete + create
try:
    client.delete_collection(collection_name="my_library")
except Exception:
    pass

client.create_collection(
    collection_name="my_library",
    vectors_config=VectorParams(
        size=1536, distance=Distance.COSINE
    ),  # Размерность как у вашей модели
)

# 2. Добавляем данные (Точки / Points)
client.upsert(
    collection_name="my_library",
    points=[
        PointStruct(
            id=1,
            vector=[0.12, 0.05, ...],
            payload={"title": "Пост про ИИ", "year": 2024},
        )
    ],
)

Payload — это дополнительные данные, которые хранятся вместе с каждым вектором. В отличие от FAISS, где метаданные поддерживаются ограниченно, Qdrant позволяет хранить произвольные JSON-структуры.

In [ ]:
from qdrant_client.models import PointStruct

points = [
    PointStruct(
        id=1,
        vector=[0.1, 0.2, 0.3, ...],  # 384-мерный вектор
        payload={
            "title": "Введение в машинное обучение",
            "author": "Иван Иванов",
            "year": 2023,
            "category": "AI",
            "tags": ["ML", "Python", "Tutorial"],
        },
    ),
    PointStruct(
        id=2,
        vector=[0.4, 0.5, 0.6, ...],
        payload={
            "title": "Глубокое обучение на практике",
            "author": "Мария Петрова",
            "year": 2024,
            "category": "Deep Learning",
            "tags": ["DL", "Neural Networks"],
        },
    ),
]

client.upsert(collection_name="my_documents", points=points)

Одна из сильнейших сторон Qdrant — мощная система фильтрации. Вы можете комбинировать семантический поиск с точными условиями по метаданным.

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchValue, Range

client = QdrantClient(url="http://localhost:6333")

# Фильтр:
# - категория = "AI"
# - год >= 2023
search_filter = Filter(
    must=[
        FieldCondition(key="category", match=MatchValue(value="AI")),
        FieldCondition(key="year", range=Range(gte=2023)),
    ]
)

# Вектор запроса
query_vector = embed_model.embed_query("машинное обучение")

# Поиск (search() удалён, используем query_points())
search_result = client.query_points(
    collection_name="my_documents",
    query=query_vector,
    query_filter=search_filter,
    limit=5,
)

# Результаты (query_points возвращает объект с атрибутом .points)
for point in search_result.points:
    print(
        {
            "id": point.id,
            "score": point.score,
            "title": point.payload.get("title"),
            "year": point.payload.get("year"),
        }
    )

### Qdrant. Интеграция с LangChain

uv add langchain-qdrant

In [ ]:
# Локальное подключение
from langchain_qdrant import QdrantVectorStore

# Создаем хранилище: LangChain сам создаст коллекцию, если её нет
vector_store = QdrantVectorStore.from_documents(
    splitted_docs,
    embed_model,
    url="http://localhost:6333",
    collection_name="my_documents",
)

# Поиск
docs_found = vector_store.similarity_search("Когда основан МГУ?", k=2)
for doc in docs_found:
    print(doc.page_content[:50])

In [ ]:
# Работа с облаком
vector_store = QdrantVectorStore.from_documents(
    splitted_docs,
    embed_model,
    url="https://your-instance.qdrant.io",
    api_key="your-api-key",
    collection_name="my_documents",
    prefer_grpc=True,  # Ускоряет передачу данных
)

In [ ]:
# Фильтрация через LangChain
from qdrant_client.models import FieldCondition, Filter, MatchValue

# Создание фильтра
search_filter = Filter(
    must=[FieldCondition(key="category", match=MatchValue(value="AI"))]
)

# Поиск с фильтром
docs = vector_store.similarity_search("машинное обучение", k=5, filter=search_filter)

In [ ]:
# Сложные фильтры
from qdrant_client.models import FieldCondition, Filter, MatchAny, MatchValue, Range

# Поиск документов:
# - категория "AI" ИЛИ "Deep Learning"
# - год между 2020 и 2024
# - есть тег "Python"
complex_filter = Filter(
    must=[
        FieldCondition(key="category", match=MatchAny(any=["AI", "Deep Learning"])),
        FieldCondition(key="year", range=Range(gte=2020, lte=2024)),
        FieldCondition(key="tags", match=MatchValue(value="Python")),
    ]
)

docs = vector_store.similarity_search("нейронные сети", k=10, filter=complex_filter)

In [ ]:
# Хранение документации разных версий продукта
payload = {
    "title": "API Reference",
    "version": "2.0",
    "section": "Authentication",
    "deprecated": False
}

# Поиск только актуальной документации
filter_current = Filter(
    must=[
        FieldCondition(key="deprecated", match=MatchValue(value=False)),
        FieldCondition(key="version", match=MatchValue(value="2.0"))
    ]
)

In [ ]:
# Метаданные с языком
payload = {
    "content": "Текст статьи...",
    "language": "ru",
    "category": "tutorial",
}

# Поиск только на русском языке
filter_russian = Filter(
    must=[FieldCondition(key="language", match=MatchValue(value="ru"))]
)

In [ ]:
import datetime

# Метаданные с датой
payload = {
    "content": "Новостная статья...",
    "published_date": "2024-01-15",
    "source": "tech_news"
}

# Поиск статей за последний месяц
one_month_ago = (datetime.datetime.now() - datetime.timedelta(days=30)).strftime("%Y-%m-%d")
filter_recent = Filter(
    must=[
        FieldCondition(
            key="published_date",
            range=Range(gte=one_month_ago)
        )
    ]
)

In [ ]:
# Для ускорения фильтрации можно создать индексы на часто используемых полях:
client.create_payload_index(
    collection_name="my_documents",
    field_name="category",
    field_schema="keyword"  # Тип индекса: keyword, integer, float, bool
)

In [ ]:
# При добавлении большого количества векторов используйте batch-операции:
# Эффективнее, чем добавлять по одному
client.upsert(
    collection_name="my_documents",
    points=large_list_of_points,  # Список из тысяч точек
    wait=True  # Дождаться завершения операции
)

Qdrant предоставляет отличный баланс между простотой и функциональностью:

- Коллекции организуют данные в логические группы с общими настройками
- Payload позволяет хранить богатые метаданные вместе с векторами
- Фильтры обеспечивают точную выборку данных с комбинированием условий
- API делает интеграцию простой и удобной
- Производительность остаётся высокой даже при сложных запросах

Qdrant - отличный выбор для production-проектов, где нужна гибкость работы с метаданными, но не требуется сложность Weaviate.